# Fase 0 — Exploración y baseline

Perfilado de fuentes Excel antes de Bronze. Entregables relacionados:
- `docs/supuestos.md`
- `docs/arquitectura.md`

Ejecutar celdas en orden. Requiere archivos en el Volume configurado abajo.

In [ ]:
%pip install openpyxl

In [ ]:
# Catálogo objetivo: ips_analytics (ver infra/ddl/00_create_catalog_schema.sql)
RAW_VOLUME_PATH = "/Volumes/ips_analytics/raw/raw_data/"

# Si aún usas la ruta del workspace inicial, descomenta:
# RAW_VOLUME_PATH = "/Volumes/workspace/ips_db/raw_data/"

SOURCE_FILES = {
    "pacientes": "pacientes.xlsx",
    "citas": "citas.xlsx",
    "eventos": "eventos_clinicos.xlsx",
    "facturacion": "facturacion.xlsx",
}

In [ ]:
import pandas as pd

df_pacientes_pd = pd.read_excel(f"{RAW_VOLUME_PATH}{SOURCE_FILES['pacientes']}")
df_citas_pd = pd.read_excel(f"{RAW_VOLUME_PATH}{SOURCE_FILES['citas']}")
df_eventos_pd = pd.read_excel(f"{RAW_VOLUME_PATH}{SOURCE_FILES['eventos']}")
df_facturacion_pd = pd.read_excel(f"{RAW_VOLUME_PATH}{SOURCE_FILES['facturacion']}")

# Exploración: strings en Spark para detectar nulos 'nan' / vacíos como en fuente
df_pacientes = spark.createDataFrame(df_pacientes_pd.astype(str))
df_citas = spark.createDataFrame(df_citas_pd.astype(str))
df_eventos = spark.createDataFrame(df_eventos_pd.astype(str))
df_facturacion = spark.createDataFrame(df_facturacion_pd.astype(str))

datasets = {
    "Pacientes": df_pacientes,
    "Citas": df_citas,
    "Eventos Clínicos": df_eventos,
    "Facturación": df_facturacion,
}

In [ ]:
for name, df in datasets.items():
    print(f"Total {name}: {df.count()}")

In [ ]:
for name, df in datasets.items():
    print(f"\n=== Esquema {name} ===")
    df.printSchema()

In [ ]:
from pyspark.sql import functions as F

def conteo_nulos(df, nombre_df):
    print(f"--- Conteo de Nulos en {nombre_df} ---")
    exprs = [
        F.count(
            F.when(F.col(c).isNull() | F.col(c).isin("nan", "None", "", "NaT"), c)
        ).alias(c)
        for c in df.columns
    ]
    df.select(*exprs).show(vertical=True)

for name, df in datasets.items():
    conteo_nulos(df, name)

In [ ]:
pk_map = {
    "Pacientes": "id_paciente",
    "Citas": "id_cita",
    "Eventos Clínicos": "id_evento",
    "Facturación": "id_factura",
}

print("--- Registros Duplicados en PK ---")
for name, pk in pk_map.items():
    df = datasets[name]
    dup = df.count() - df.select(pk).distinct().count()
    print(f"{name} ({pk}): {dup}")

In [ ]:
citas_huerfanas = df_citas.join(df_pacientes, "id_paciente", "left_anti").count()
eventos_sin_cita = df_eventos.join(df_citas, "id_cita", "left_anti").count()
eventos_sin_pac = df_eventos.join(df_pacientes, "id_paciente", "left_anti").count()
facturas_sin_cita = df_facturacion.join(df_citas, "id_cita", "left_anti").count()

df_fact_check = (
    df_facturacion
    .withColumn("bruto", F.col("valor_bruto").cast("double"))
    .withColumn("desc", F.col("valor_descuento").cast("double"))
    .withColumn("neto", F.col("valor_neto").cast("double"))
)
inconsistencias_finanzas = df_fact_check.filter(
    F.round(F.col("bruto") - F.col("desc"), 2) != F.round(F.col("neto"), 2)
).count()

evt_cita = df_eventos.alias("e").join(
    df_citas.select("id_cita", "id_paciente").alias("c"), "id_cita"
)
paciente_inconsistente = evt_cita.filter(F.col("e.id_paciente") != F.col("c.id_paciente")).count()

print("--- Integridad Referencial y Reglas de Negocio ---")
print(f"Citas sin paciente válido (Huérfanas): {citas_huerfanas}")
print(f"Eventos sin cita válida (Huérfanos): {eventos_sin_cita}")
print(f"Eventos sin paciente válido (Huérfanos): {eventos_sin_pac}")
print(f"Facturas sin cita válida (Huérfanas): {facturas_sin_cita}")
print(f"Eventos con id_paciente distinto al de la cita: {paciente_inconsistente}")
print(f"Facturas con inconsistencia (bruto - desc != neto): {inconsistencias_finanzas}")

In [ ]:
CARDINALIDAD = {
    "Pacientes": ["id_paciente", "sexo", "aseguradora", "estado", "ciudad"],
    "Citas": ["id_cita", "id_paciente", "id_profesional", "especialidad", "sede", "tipo_cita", "estado_cita"],
    "Eventos Clínicos": ["id_evento", "id_cita", "id_paciente", "tipo_evento", "estado_evento"],
    "Facturación": ["id_factura", "id_paciente", "id_cita", "tipo_servicio", "pagador", "estado_pago"],
}

for name, cols in CARDINALIDAD.items():
    df = datasets[name]
    print(f"\n--- {name} ---")
    for c in cols:
        print(f"  {c}: {df.select(c).distinct().count()} distinct")

In [ ]:
def top_values(df, column, n=10):
    print(f"  {column}:")
    df.groupBy(column).count().orderBy(F.desc("count")).show(n, truncate=False)

print("=== Pacientes ===")
for col in ["sexo", "aseguradora", "estado"]:
    top_values(df_pacientes, col)

print("=== Citas ===")
for col in ["estado_cita", "tipo_cita", "especialidad"]:
    top_values(df_citas, col, 15)

print("=== Eventos ===")
for col in ["tipo_evento", "estado_evento"]:
    top_values(df_eventos, col)

print("=== Facturación ===")
for col in ["estado_pago", "tipo_servicio"]:
    top_values(df_facturacion, col)

In [ ]:
def rango_timestamp(df, col_name, label):
    parsed = df.withColumn("_ts", F.to_timestamp(col_name))
    row = parsed.select(F.min("_ts").alias("min"), F.max("_ts").alias("max")).collect()[0]
    print(f"{label}: {row['min']} -> {row['max']}")

rango_timestamp(df_citas, "fecha_hora_cita", "Citas fecha_hora_cita")
rango_timestamp(df_eventos, "fecha_hora_evento", "Eventos fecha_hora_evento")
rango_timestamp(df_facturacion, "fecha_factura", "Facturación fecha_factura")
rango_timestamp(df_pacientes, "fecha_nacimiento", "Pacientes fecha_nacimiento (to_timestamp)")

In [ ]:
total_citas = df_citas.count()
citas_con_evento = df_eventos.select("id_cita").distinct().count()
citas_con_factura = df_facturacion.select("id_cita").distinct().count()
pacientes_con_cita = df_citas.select("id_paciente").distinct().count()
total_pacientes = df_pacientes.count()

print("--- Cobertura ---")
print(f"Pacientes con al menos una cita: {pacientes_con_cita} / {total_pacientes}")
print(f"Citas con al menos un evento: {citas_con_evento} / {total_citas}")
print(f"Citas con al menos una factura: {citas_con_factura} / {total_citas}")
print(f"Promedio citas por paciente (con cita): {round(total_citas / pacientes_con_cita, 2)}")

In [ ]:
# Detección fuera de Spark para tipos mixtos en Excel
tipos_fn = df_pacientes_pd["fecha_nacimiento"].map(lambda x: type(x).__name__)
print("Tipos en fecha_nacimiento (pacientes):")
print(tipos_fn.value_counts().to_string())
valid_types = {"datetime", "Timestamp", "date"}
mask_invalido = ~tipos_fn.isin(valid_types)
if mask_invalido.any():
    print("\nRegistros con tipo inesperado (revisar en Silver):")
    display(df_pacientes_pd.loc[mask_invalido, ["id_paciente", "fecha_nacimiento"]])

In [ ]:
MUESTRA_COLS = {
    "Pacientes": ["id_paciente", "tipo_documento", "sexo", "aseguradora", "ciudad", "estado"],
    "Citas": ["id_cita", "id_paciente", "especialidad", "sede", "fecha_hora_cita", "tipo_cita", "estado_cita"],
    "Eventos Clínicos": ["id_evento", "id_cita", "id_paciente", "tipo_evento", "fecha_hora_evento", "estado_evento"],
    "Facturación": ["id_factura", "id_paciente", "id_cita", "tipo_servicio", "fecha_factura", "valor_neto", "estado_pago"],
}

for name, cols in MUESTRA_COLS.items():
    print(f"\n=== Muestra {name} (5 filas) ===")
    datasets[name].select(cols).show(5, truncate=False)

## Cierre Fase 0

Tras ejecutar este notebook:
1. Verificar que los conteos coinciden con `docs/supuestos.md`.
2. Ejecutar `infra/ddl/00_create_catalog_schema.sql` si aún no existe el catálogo.
3. Continuar con **Fase 1** (contratos de datos y matriz de calidad).